# Data Lakehouse Platform - Quick Start Notebook

This notebook demonstrates the basic operations of the data lakehouse platform including:
- Spark initialization with Iceberg and Nessie
- Database and table creation
- Data ingestion and querying
- Time travel capabilities

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '/workspace')

from src.core import SparkSessionBuilder, CatalogManager, LakehouseConfig
from src.ingestion.data_ingestion import DataIngestionService
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Imports successful")

## 2. Initialize Spark Session

In [ ]:
# Build Spark session with Iceberg and Nessie support
spark = SparkSessionBuilder.build(
    app_name="quickstart_notebook",
    master="spark://spark-master:7077",
    use_nessie=True,
    use_iceberg_rest=False
)

print("✓ Spark session initialized")
print(f"Spark Version: {spark.version}")

## 3. Create Database and Tables

In [ ]:
# Initialize catalog manager
catalog = CatalogManager(spark, catalog_name="nessie")

# Create database
catalog.create_database("quickstart", location="s3a://warehouse/quickstart/")
print("✓ Database created: nessie.quickstart")

# Create users table
catalog.create_table(
    database_name="quickstart",
    table_name="users",
    schema_str="id INT, name STRING, email STRING, age INT, created_at TIMESTAMP",
    partitioning="age"
)
print("✓ Table created: nessie.quickstart.users")

## 4. Insert Sample Data

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from datetime import datetime

# Create sample data
data = [
    (1, "Alice", "alice@example.com", 30, datetime(2024, 1, 1)),
    (2, "Bob", "bob@example.com", 25, datetime(2024, 1, 2)),
    (3, "Charlie", "charlie@example.com", 35, datetime(2024, 1, 3)),
    (4, "Diana", "diana@example.com", 28, datetime(2024, 1, 4)),
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("created_at", TimestampType(), True),
])

df = spark.createDataFrame(data, schema=schema)

# Write to table
df.write.mode("append").format("iceberg") \
    .option("write.format.default", "parquet") \
    .saveAsTable("nessie.quickstart.users")

print(f"✓ Inserted {len(data)} records")

## 5. Query Data

In [ ]:
# Query all data
users_df = spark.sql("SELECT * FROM nessie.quickstart.users")
print(f"Total records: {users_df.count()}")
users_df.show()

# Filter query
print("\nUsers over 28:")
spark.sql("SELECT * FROM nessie.quickstart.users WHERE age > 28").show()

## 6. Update Data

In [ ]:
# Update data (append new records)
new_data = [
    (5, "Eve", "eve@example.com", 32, datetime(2024, 1, 5)),
    (6, "Frank", "frank@example.com", 29, datetime(2024, 1, 6)),
]

new_df = spark.createDataFrame(new_data, schema=schema)
new_df.write.mode("append").format("iceberg") \
    .option("write.format.default", "parquet") \
    .saveAsTable("nessie.quickstart.users")

print(f"✓ Added {len(new_data)} new records")
print(f"New total: {spark.sql('SELECT COUNT(*) FROM nessie.quickstart.users').collect()[0][0]} records")

## 7. Data Validation

In [ ]:
from src.ingestion.data_ingestion import DataIngestionService

ingestion = DataIngestionService(spark, catalog_name="nessie")

# Validate data quality
report = ingestion.validate_data_quality(
    database_name="quickstart",
    table_name="users",
    null_checks={"id": True, "name": True, "email": True},
    unique_checks=["email"]
)

print("Data Quality Report:")
print(f"  Total Records: {report['total_records']}")
print(f"  Validation Status: {'✓ VALID' if report['is_valid'] else '✗ INVALID'}")
print(f"  Null Checks: {report['null_checks']}")
print(f"  Unique Checks: {report['unique_checks']}")

## 8. Schema Information

In [ ]:
# Describe table
spark.sql("DESCRIBE nessie.quickstart.users").show()

# Get detailed schema
df = spark.read.table("nessie.quickstart.users")
print("\nDetailed Schema:")
df.printSchema()

## 9. Aggregation and Analytics

In [ ]:
# Aggregation examples
print("Age Statistics:")
spark.sql("""
    SELECT 
        COUNT(*) as total_users,
        AVG(age) as avg_age,
        MIN(age) as min_age,
        MAX(age) as max_age
    FROM nessie.quickstart.users
""").show()

print("\nAge Distribution:")
spark.sql("""
    SELECT 
        CASE 
            WHEN age < 25 THEN '20-24'
            WHEN age < 30 THEN '25-29'
            WHEN age < 35 THEN '30-34'
            ELSE '35+'
        END as age_group,
        COUNT(*) as count
    FROM nessie.quickstart.users
    GROUP BY age_group
    ORDER BY age_group
""").show()

## 10. Summary

In [ ]:
print("="*50)
print("✓ Quickstart Completed!")
print("="*50)
print("\nYou have successfully:")
print("  1. Initialized Spark with Iceberg and Nessie")
print("  2. Created a database and table in Iceberg")
print("  3. Ingested sample data")
print("  4. Queried and analyzed the data")
print("  5. Updated table with new records")
print("  6. Validated data quality")
print("\nNext Steps:")
print("  - Explore Dremio UI at http://localhost:9047")
print("  - Check MLflow at http://localhost:5000")
print("  - Document data in OpenMetadata at http://localhost:8585")
print("\nFor more information, see docs/DEVELOPMENT.md")